# S3 usage (public bucket)

In [1]:
%pip install boto3
%env AWS_ENDPOINT_URL=https://s3.mesocentre.uca.fr
%env AWS_S3_ENDPOINT=s3.mesocentre.uca.fr
%env AWS_REGION=fr-clermont-mesocentre

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
env: AWS_ENDPOINT_URL=https://s3.mesocentre.uca.fr
env: AWS_S3_ENDPOINT=s3.mesocentre.uca.fr
env: AWS_REGION=fr-clermont-mesocentre


In [2]:
import boto3
import botocore

# Create unsigned AWS S3 client
s3 = boto3.resource('s3', config=botocore.config.Config(signature_version=botocore.UNSIGNED))

notebook_bucket = s3.Bucket("eosc-fairease-notebook-public")

In [3]:
# List objects in bucket (max 1000 objects)
for obj in notebook_bucket.objects.all():
    print(obj.key, " - ", obj.last_modified)

GLODAPv2.2023_Merged_Master_File.csv.zip  -  2025-06-11 14:08:33.489000+00:00
glodap_v2_2023_merged_master.parquet  -  2025-06-11 15:02:51.442000+00:00


In [4]:
# Get Metadata of an object
obj = notebook_bucket.Object('GLODAPv2.2023_Merged_Master_File.csv.zip')
print(type(obj))
print(obj.content_type, " ", obj.content_length, "bytes")
print(obj.metadata)

<class 'boto3.resources.factory.s3.Object'>
application/zip   110353886 bytes
{'eosc-fairease-status': 'demo'}


In [5]:
import zipfile
from io import BytesIO

# Get Objet in memory
body = obj.get()["Body"]
buffer = BytesIO(body.read())

z = zipfile.ZipFile(buffer)
for filename in z.namelist():
    file_info = z.getinfo(filename)
    print(file_info)
    with z.open('GLODAPv2.2023_Merged_Master_File.csv') as csvfile:
        print(csvfile.readline())
        ##

<ZipInfo filename='GLODAPv2.2023_Merged_Master_File.csv' compress_type=deflate filemode='-rw-rw-r--' file_size=894645727 compress_size=110353664>
b'G2expocode,G2cruise,G2station,G2region,G2cast,G2year,G2month,G2day,G2hour,G2minute,G2latitude,G2longitude,G2bottomdepth,G2maxsampdepth,G2bottle,G2pressure,G2depth,G2temperature,G2theta,G2salinity,G2salinityf,G2salinityqc,G2sigma0,G2sigma1,G2sigma2,G2sigma3,G2sigma4,G2gamma,G2oxygen,G2oxygenf,G2oxygenqc,G2aou,G2aouf,G2nitrate,G2nitratef,G2nitrateqc,G2nitrite,G2nitritef,G2silicate,G2silicatef,G2silicateqc,G2phosphate,G2phosphatef,G2phosphateqc,G2tco2,G2tco2f,G2tco2qc,G2talk,G2talkf,G2talkqc,G2fco2,G2fco2f,G2fco2temp,G2phts25p0,G2phts25p0f,G2phtsinsitutp,G2phtsinsitutpf,G2phtsqc,G2cfc11,G2pcfc11,G2cfc11f,G2cfc11qc,G2cfc12,G2pcfc12,G2cfc12f,G2cfc12qc,G2cfc113,G2pcfc113,G2cfc113f,G2cfc113qc,G2ccl4,G2pccl4,G2ccl4f,G2ccl4qc,G2sf6,G2psf6,G2sf6f,G2sf6qc,G2c13,G2c13f,G2c13qc,G2c14,G2c14f,G2c14err,G2h3,G2h3f,G2h3err,G2he3,G2he3f,G2he3err,G2he,G2hef,G2

## With Pandas

In [6]:
%pip install pandas geopandas "s3fs<=0.4" pyarrow

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
## Read directly zipped csv with pandas on a public S3 bucket

import pandas as pd
import geopandas as gpd
import s3fs
s3 = s3fs.S3FileSystem(anon=True)

df = pd.read_csv(s3.open("s3://eosc-fairease-notebook-public/GLODAPv2.2023_Merged_Master_File.csv.zip"), compression='zip', dtype={'G2expocode' : str, 'G2Cruise' : int, 'G2region' :int , 'G2cast' :int ,'G2year': int,'G2month' : int,'G2day' : int, 'G2hour': int, 'G2minute' : int, 'G2bottle' : int, 'G2doi' :  str})
pd.set_option('display.max_columns', None) 
df.head(10)
df.info()

# Remove the first 2 characters from column names
df.columns = df.columns.str[2:]

# Identify columns ending with 'qc', these look like int datatypes also
qc_columns = [col for col in df.columns if col.endswith('qc')]
df[qc_columns] = df[qc_columns].astype(int)  

# Combine year, month, day, hour, and minute columns into a new datetime 'recordedtime' column, would be useful for time queries
df['recordedtime'] = pd.to_datetime(df[['year', 'month', 'day', 'hour', 'minute']])

from shapely.geometry import Point
gdf = gpd.GeoDataFrame(df,
                        geometry=gpd.points_from_xy(df['longitude'], df['latitude']),
                        crs='EPSG:4326')  # Set the coordinate reference system (CRS)

# Save file locally
gdf.to_parquet('glodap_v2_2023_merged_master.parquet')
print("done")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1402829 entries, 0 to 1402828
Columns: 109 entries, G2expocode to G2doi
dtypes: float64(99), int64(8), object(2)
memory usage: 1.1+ GB
done


# With DuckDB

In [8]:
%pip install duckdb

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [19]:
import duckdb
r = duckdb.sql("""CREATE SECRET IF NOT EXISTS secret (TYPE S3,PROVIDER CREDENTIAL_CHAIN,CHAIN 'env',ENDPOINT 's3.mesocentre.uca.fr');""")
r = duckdb.sql("""SELECT COUNT(*) FROM read_parquet('s3://eosc-fairease-notebook-public/glodap_v2_2023_merged_master.parquet');""")
print(r)
r = duckdb.sql("""SELECT doi, MAX(temperature) FROM read_parquet('s3://eosc-fairease-notebook-public/glodap_v2_2023_merged_master.parquet') GROUP BY doi ORDER BY MAX(temperature) DESC;""")
print(r)


┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│      1402829 │
└──────────────┘

┌───────────────────────────────────────────────────────────────────────┬──────────────────┐
│                                  doi                                  │ max(temperature) │
│                                varchar                                │      double      │
├───────────────────────────────────────────────────────────────────────┼──────────────────┤
│ https://doi.org/10.25921/16y6-9e29                                    │           34.501 │
│ https://doi.org/10.25921/0sta-y820                                    │            32.68 │
│ https://doi.org/10.3334/cdiac/otg.ndp080                              │           32.171 │
│ https://doi.org/10.3334/cdiac/otg.pacifica_49hg19930807               │             31.8 │
│ https://doi.org/10.25921/f4vg-g356;https://doi.org/10.25921/531n-c230 │          31.7353 │
│ https://doi.org/10.25921/cfqr-8657                        